In [19]:
import os
import json
import numpy as np
import pandas as pd
from obspy import read
from datetime import datetime, timedelta


# ==========================
# KONFIGURASI
# ==========================

CATALOG_CSV = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog_CLEAN.csv'
WAVEFORM_ROOT = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02'
OUTPUT_JSON = "/Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json"


SAMPLE_RATE = 100
WINDOW_SEC = 7
WINDOW_SAMPLES = SAMPLE_RATE * WINDOW_SEC


# ==========================
# UTIL
# ==========================

def load_catalog(csv_path):
    df = pd.read_csv(csv_path)
    df["origin_dt"] = pd.to_datetime(df["origin_time"], utc=True)
    return df


def extract_event_time_from_path(path):
    from datetime import timezone
    folder = os.path.basename(os.path.dirname(path))
    ts_str = folder.split("-")[1]  # 20130103192247
    return datetime.strptime(ts_str, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)


def match_to_catalog(catalog_df, event_time, tolerance_sec=10):
    deltas = (catalog_df["origin_dt"] - event_time).abs()
    idx = deltas.argmin()
    if deltas.iloc[idx].total_seconds() <= tolerance_sec:
        return catalog_df.iloc[idx]
    return None


def iter_waveforms(root):
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(".mseed"):
                yield os.path.join(dp, fn)


def extract_window(trace, center_time, sr=SAMPLE_RATE, win_sec=WINDOW_SEC):
    start = center_time - timedelta(seconds=win_sec/2)
    end = center_time + timedelta(seconds=win_sec/2)

    st = trace.slice(starttime=start, endtime=end)

    # Jika slice kosong → kembalikan array nol
    if len(st) == 0:
        return [0.0] * WINDOW_SAMPLES

    # Ambil data
    raw = st[0].data

    # Paksa menjadi numpy array dengan shape
    try:
        data = np.asarray(raw, dtype=float)
    except:
        # Jika tetap gagal → fallback
        return [0.0] * WINDOW_SAMPLES

    # Jika data tidak punya panjang (scalar)
    if data.ndim == 0:
        return [float(data)] + [0.0] * (WINDOW_SAMPLES - 1)

    # Pastikan panjang 700 sampel
    if len(data) < WINDOW_SAMPLES:
        data = np.pad(data, (0, WINDOW_SAMPLES - len(data)))
    else:
        data = data[:WINDOW_SAMPLES]

    return data.tolist()




def extract_noise_window(trace, center_time, offset_sec=20):
    """
    Noise diambil 20 detik setelah event.
    """
    noise_center = center_time + timedelta(seconds=offset_sec)
    return extract_window(trace, noise_center)


# ==========================
# MAIN
# ==========================

def build_json():
    catalog = load_catalog(CATALOG_CSV)
    output = {}

    for wf_path in iter_waveforms(WAVEFORM_ROOT):
        try:
            event_time = extract_event_time_from_path(wf_path)
        except:
            continue

        matched = match_to_catalog(catalog, event_time)
        if matched is None:
            continue

        try:
            st = read(wf_path)
            tr = st[0]
        except:
            continue

        # ambil window seismic

        # offset event ke trace (keduanya tz-aware)
        offset = event_time - tr.stats.starttime.datetime.replace(tzinfo=event_time.tzinfo)

        # waktu pusat window seismic
        center_time = tr.stats.starttime + offset

        Z = extract_window(tr, center_time)
        Z_noise = extract_noise_window(tr, center_time)

        event_id = matched["Event ID"]

        output[event_id] = {
            "type": "se",
            "Z": Z,
            "Z_noise": Z_noise
        }

                
    with open(OUTPUT_JSON, "w") as f:
        json.dump(output, f, indent=2)

    print(f"Selesai! JSON disimpan ke {OUTPUT_JSON}")


if __name__ == "__main__":
    build_json()


Selesai! JSON disimpan ke /Volumes/Extreme SSD/stream_indonesia_waveform/Indonesia_MCU_Quake_Zonly.json
